# Exercise 30.2c solution


In [1]:
# --- Setup code from previous sub-exercises ---
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Model parameters
R_T = 1
k_0_on = 0
k_0_off = 100
k_Ca_on = 120
k_Ca_off = 50
Ca_50 = k_Ca_off / k_Ca_on

f_0 = 50
f_prime_0 = 400
h = 8
h_prime = 6
g = 4

u = 8
v = 2
w = 3

Ca = Ca_50 * 100


def rhs_cooperative(t, y, Ca, u, v, w):
    """Right-hand side of the cooperative Razumova model."""
    D, A_1, A_2 = y

    # Ca-dependent base rates
    k_u_on = k_0_on + (k_Ca_on - k_0_on) * Ca / (Ca_50 + Ca)
    k_u_off = k_0_off + (k_Ca_off - k_0_off) * Ca / (Ca_50 + Ca)

    # State subpopulations
    R_off = R_T - D - A_1 - A_2
    lambda_A2 = A_2 / R_T
    lambda_on = (D + A_1 + A_2) / R_T

    # XB-XB cooperativity (parameter v)
    f = f_0 * (1 + lambda_A2 * (np.exp(v - 1) - 1)) ** 2
    f_prime = f_prime_0 * (1 + lambda_A2 * (np.exp(-(v - 1)) - 1)) ** 2

    # RU-RU cooperativity (parameter u)
    k_w_on = k_u_on * (1 + lambda_on * (u - 1)) ** 2
    k_w_off = k_u_off * (u - lambda_on * (u - 1)) ** 2

    # XB-RU cooperativity (parameter w)
    k_on = k_w_on * (1 + lambda_A2 * (np.exp(w - 1) - 1)) ** 2
    k_off = k_w_off * (1 + lambda_A2 * (np.exp(-(w - 1)) - 1)) ** 2

    # ODEs
    dD_dt = k_on * R_off + f_prime * A_1 + g * A_2 - (k_off + f) * D
    dA1_dt = f * D + h_prime * A_2 - (f_prime + h) * A_1
    dA2_dt = h * A_1 - (h_prime + g) * A_2

    return [dD_dt, dA1_dt, dA2_dt]


# ----------------------------------------------

In [2]:
import ipywidgets as widgets


def cooperativity_widget(u=8, v=2, w=3):
    """Interactive force-pCa curve with adjustable cooperativity."""
    Ca_values = np.logspace(-2, 2, 30) * Ca_50
    ss_force = []

    for Ca_val in Ca_values:
        sol = solve_ivp(
            rhs_cooperative,
            (0, 50),
            [0.01, 0.01, 0.01],
            method="LSODA",
            args=(Ca_val, u, v, w),
        )
        ss_force.append(sol.y[2, -1])

    ss_force = np.array(ss_force)
    pCa_vals = -np.log10(Ca_values)

    # Also solve at highest Ca for force development
    t_eval_fd = np.linspace(0, 2, 500)
    sol_fd = solve_ivp(
        rhs_cooperative,
        (0, 2),
        [0.01, 0.01, 0.01],
        t_eval=t_eval_fd,
        method="LSODA",
        args=(Ca_values[-1], u, v, w),
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

    # Force-pCa
    f_max = ss_force.max()
    if f_max > 0:
        axes[0].plot(pCa_vals, ss_force / f_max, "o-", color="C0")
    else:
        axes[0].plot(pCa_vals, ss_force, "o-", color="C0")
    axes[0].set(
        xlabel="pCa",
        ylabel="Normalised force",
        title=f"Force-pCa (u={u}, v={v}, w={w})",
        ylim=(-0.05, 1.1),
    )
    axes[0].invert_xaxis()

    # Force development
    axes[1].plot(sol_fd.t, sol_fd.y[2], color="C3")
    axes[1].set(
        xlabel="Time (s)",
        ylabel="Relative force ($A_2$)",
        title="Force development (high Ca)",
    )

    plt.show()


widgets.interact(
    cooperativity_widget,
    u=widgets.FloatSlider(value=8, min=1, max=20, step=0.5, description="u (RU-RU)"),
    v=widgets.FloatSlider(value=2, min=1, max=5, step=0.25, description="v (XB-XB)"),
    w=widgets.FloatSlider(value=3, min=1, max=8, step=0.25, description="w (XB-RU)"),
);

interactive(children=(FloatSlider(value=8.0, description='u (RU-RU)', max=20.0, min=1.0, step=0.5), FloatSlide…

1. What is the minimum value of $u$ needed to produce a steep, sigmoidal curve that resembles the experimental data?

Values of $u$ around 6–10 produce physiologically realistic steepness (Hill coefficient ≈ 3–7). Below $u \approx 4$, the curve is too shallow to match experimental data. The parameter $u$ primarily controls the Hill coefficient (steepness) of the force-pCa curve, because it governs how much an activated neighbouring regulatory unit facilitates the activation of its neighbour (RU-RU cooperativity).

2. How does the $\text{pCa}_{50}$ shift when you increase $w$? Does the muscle become more or less sensitive to calcium?

Increasing $w$ shifts the curve to the left (toward higher pCa / lower calcium concentrations), meaning the system becomes more calcium-sensitive. This is because XB-RU cooperativity causes strongly bound crossbridges to hold the regulatory units in the open (permissive) conformation, effectively reducing the calcium concentration needed for activation.

3. Set all parameters to 1 (no cooperativity). How does the resulting curve compare to experimental cardiac muscle data? What does this tell you about the role of cooperativity?

Without cooperativity ($u=v=w=1$), the force-pCa curve is shallow and hyperbolic with a Hill coefficient of approximately 1. This is far too gradual compared to the steep sigmoidal relationship observed experimentally. This demonstrates that cooperativity is essential for reproducing the switch-like calcium sensitivity of real muscle — independent regulatory units acting alone cannot explain the observed behaviour.
